# GéoMarketing IDF : J10e — Sports, loisirs et culture en Île-de-France

## Objectifs
- Extraire le domaine F de la BPE 2025.
- Conserver les observations détaillées pour la cartographie.
- Calculer les nombres par groupe et sous-catégorie.
- Calculer les parts des groupes et les parts intragroupe.
- Calculer les densités uniquement par groupe.
- Examiner les capacités des cinémas.
- Enrichir le profil communal J10d.

## Unité de comptage
Une observation BPE, pas nécessairement un bâtiment ou un équipement unique.

## Limites
- Pas de mesure de fréquentation.
- Pas de confirmation d'ouverture actuelle.
- Les capacités des cinémas correspondent à des fauteuils, pas à des entrées.
- La multiplicité des équipements sportifs reste distincte du nombre d'observations.
- Les accueils de loisirs D505 sont déjà traités au J10d.

In [1]:
#Import librairies

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les chemins

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10e"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = (
    RACINE / "data" / "raw" / "bpe" / "BPE25.csv"
)

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10d.csv"
)

FICHIER_METADATA_J10D = (
    RACINE / "data" / "interim" / "j10d" / "metadata_j10d.json"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "sports_loisirs_culture_idf_j10e.csv"
)

FICHIER_CINEMAS = (
    DOSSIER_PROCESSED / "cinemas_idf_j10e.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "sports_loisirs_culture_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10e.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Entrée :", FICHIER_PROFIL)
print("Sortie :", FICHIER_SORTIE)

Entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10d.csv
Sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10e.csv


In [3]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [4]:
#Charger le profil J10d

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux manquants ou invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Plusieurs lignes existent pour une commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Le profil contient des codes hors IDF.")

if "NB_OBS_SANTE_ACTION_SOCIALE_BPE2025" not in profil.columns:
    raise ValueError("Le total attendu du J10d est absent.")

codes_profil = set(profil["CODGEO"])

print("Communes :", len(profil))
print("Colonnes initiales :", len(profil.columns))
display(profil.head())

Communes : 1266
Colonnes initiales : 783


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,PART_OTO_RHINO_LARYNGOLOGIE_DANS_GROUPE_PCT,PART_PEDIATRIE_DANS_GROUPE_PCT,PART_PNEUMOLOGIE_DANS_GROUPE_PCT,PART_RADIODIAGNOSTIC_IMAGERIE_MEDICALE_DANS_GROUPE_PCT,PART_STOMATOLOGIE_DANS_GROUPE_PCT,PART_GYNECOLOGIE_OBSTETRIQUE_DANS_GROUPE_PCT,PART_CHIRURGIENS_DENTISTES_DANS_GROUPE_PCT,PART_SAGES_FEMMES_DANS_GROUPE_PCT,PART_MASSEURS_KINESITHERAPEUTES_DANS_GROUPE_PCT,PART_PEDICURES_PODOLOGUES_DANS_GROUPE_PCT,PART_INFIRMIERS_DANS_GROUPE_PCT,PART_ORTHOPHONISTES_DANS_GROUPE_PCT,PART_ORTHOPTISTES_DANS_GROUPE_PCT,PART_LABORATOIRES_BIOLOGIE_MEDICALE_DANS_GROUPE_PCT,PART_AMBULANCES_DANS_GROUPE_PCT,PART_TRANSFUSION_SANGUINE_DANS_GROUPE_PCT,PART_ETABLISSEMENTS_THERMAUX_DANS_GROUPE_PCT,PART_PHARMACIES_DANS_GROUPE_PCT,PART_HEBERGEMENT_PERSONNES_AGEES_DANS_GROUPE_PCT,PART_SOINS_DOMICILE_PERSONNES_AGEES_DANS_GROUPE_PCT,PART_AIDE_PERSONNES_AGEES_DANS_GROUPE_PCT,PART_ACCUEIL_JEUNES_ENFANTS_DANS_GROUPE_PCT,PART_LIEUX_ACCUEIL_ENFANTS_PARENTS_DANS_GROUPE_PCT,PART_RELAIS_PETITE_ENFANCE_DANS_GROUPE_PCT,PART_ACCUEILS_LOISIRS_SANS_HEBERGEMENT_DANS_GROUPE_PCT,PART_CENTRES_SOCIAUX_DANS_GROUPE_PCT,PART_MEDIATION_FAMILIALE_DANS_GROUPE_PCT,PART_ESPACES_VIE_SOCIALE_DANS_GROUPE_PCT,PART_MICRO_CRECHES_FINANCEMENT_PAJE_DANS_GROUPE_PCT,PART_HEBERGEMENT_ENFANTS_HANDICAP_DANS_GROUPE_PCT,PART_SERVICES_ENFANTS_HANDICAP_DANS_GROUPE_PCT,PART_ACCUEIL_HEBERGEMENT_ADULTES_HANDICAP_DANS_GROUPE_PCT,PART_AIDE_ADULTES_HANDICAP_DANS_GROUPE_PCT,PART_TRAVAIL_PROTEGE_DANS_GROUPE_PCT,PART_SOINS_DOMICILE_ADULTES_HANDICAP_DANS_GROUPE_PCT,PART_AIDE_HANDICAP_VISUEL_AUDITIF_DANS_GROUPE_PCT,PART_HEBERGEMENT_PROTECTION_ENFANCE_DANS_GROUPE_PCT,PART_ACTION_EDUCATIVE_PROTECTION_ENFANCE_DANS_GROUPE_PCT,PART_HEBERGEMENT_REINSERTION_SOCIALE_DANS_GROUPE_PCT,PART_HEBERGEMENT_PROVISOIRE_DANS_GROUPE_PCT,PART_ACCUEIL_DEMANDEURS_ASILE_DANS_GROUPE_PCT,PART_AUTRES_HEBERGEMENTS_PUBLICS_DIFFICULTE_DANS_GROUPE_PCT,PART_AIDE_FEMMES_VULNERABILITE_DANS_GROUPE_PCT,DENSITE_OBS_STRUCTURES_SOINS_10000_HAB,DENSITE_OBS_PROFESSIONNELS_SANTE_RECENSES_10000_HAB,DENSITE_OBS_PHARMACIES_SERVICES_SANITAIRES_10000_HAB,DENSITE_OBS_SERVICES_PERSONNES_AGEES_10000_HAB,DENSITE_OBS_ENFANCE_FAMILLES_VIE_SOCIALE_10000_HAB,DENSITE_OBS_ACCOMPAGNEMENT_HANDICAP_10000_HAB,DENSITE_OBS_PROTECTION_HEBERGEMENT_SOCIAL_10000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,0.7470355731225297,0.9486166007905139,0.28063241106719367,4.284584980237154,0.46245059288537554,2.359683794466403,9.667984189723

In [5]:
#Définir les groupes et sous-catégories

GROUPES = {
    "F1": "SPORTS",
    "F2": "LOISIRS_PLEIN_AIR",
    "F3": "CULTURE",
}

TYPES = {
    # Sports
    "F101": "NATATION",
    "F102": "BOULES_PETANQUE",
    "F103": "TENNIS",
    "F105": "SKI",
    "F106": "EQUITATION",
    "F107": "ATHLETISME",
    "F108": "GOLF",
    "F109": "PARCOURS_SPORT_SANTE",
    "F110": "SPORTS_GLACE",
    "F111": "JEUX_SPORTIFS_EXTERIEURS",
    "F113": "TERRAINS_GRANDS_SPORTS",
    "F114": "SPORTS_COMBAT",
    "F116": "SALLES_SPORTIVES_NON_SPECIALISEES",
    "F118": "SPORTS_NAUTIQUES",
    "F119": "BOWLING",
    "F120": "FITNESS_REMISE_EN_FORME",
    "F121": "GYMNASES_MULTISPORTS",
    "F122": "SPORTS_MECANIQUES",
    "F123": "MURS_FRONTONS",
    "F124": "TIR",
    "F125": "SPORTS_AERIENS",
    "F126": "MODELISME",
    "F127": "ESCALADE_ARTIFICIELLE",
    "F128": "CYCLISME",
    "F129": "SALLES_SPORTIVES_SPECIALISEES",
    "F130": "SKATE_ROLLER_VELO_FREESTYLE",

    # Loisirs de plein air
    "F201": "BAIGNADE_AMENAGEE",
    "F202": "PLAISANCE_MOUILLAGE",
    "F203": "RANDONNEE_ORIENTATION",
    "F204": "SPORTS_NATURE",

    # Culture
    "F303": "CINEMAS",
    "F305": "CONSERVATOIRES",
    "F307": "BIBLIOTHEQUES",
    "F312": "EXPOSITION_MEDIATION_CULTURELLE",
    "F313": "PATRIMOINE_ESPACES_REMARQUABLES",
    "F314": "ARCHIVES",
    "F315": "ARTS_SPECTACLE",
}

assert len(TYPES.values()) == len(set(TYPES.values()))
assert not (set(TYPES.values()) & set(GROUPES.values()))

referentiel = pd.DataFrame({
    "TYPEQU": list(TYPES),
    "CATEGORIE_ANALYSE": list(TYPES.values()),
})

referentiel["GROUPE_CODE"] = referentiel["TYPEQU"].str[:2]
referentiel["GROUPE_ANALYSE"] = (
    referentiel["GROUPE_CODE"].map(GROUPES)
)

sauvegarder_csv(
    referentiel,
    DOSSIER_INTERIM / "referentiel_sports_loisirs_culture.csv",
)

display(referentiel)

,TYPEQU,CATEGORIE_ANALYSE,GROUPE_CODE,GROUPE_ANALYSE
0,F101,NATATION,F1,SPORTS
1,F102,BOULES_PETANQUE,F1,SPORTS
2,F103,TENNIS,F1,SPORTS
3,F105,SKI,F1,SPORTS
4,F106,EQUITATION,F1,SPORTS
5,F107,ATHLETISME,F1,SPORTS
6,F108,GOLF,F1,SPORTS
7,F109,PARCOURS_SPORT_SANTE,F1,SPORTS
8,F110,SPORTS_GLACE,F1,SPORTS
9,F111,JEUX_SPORTIFS_EXTERIEURS,F1,SPORTS


In [6]:
#Vérifier l'en-tete BPE

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = [
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    ]

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "En-tête BPE détaillé non reconnu : "
        "AN, DEPCOM et TYPEQU sont attendus."
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms = [
    normaliser_nom_colonne(colonne)
    for colonne in entete.columns
]

if len(noms) != len(set(noms)):
    raise ValueError("Colonnes ambiguës après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))

Séparateur : ';'


In [7]:
#Extraire les observations par blocs

blocs_retenus = []
nombre_lignes_lues = 0
codes_inconnus = set()

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(c) for c in bloc.columns
        ]

        bloc["RANG_SOURCE_J10E"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        est_idf = communes.str[:2].isin(DEPARTEMENTS_IDF)
        est_domaine_f = types.str.startswith("F", na=False)

        inconnus = est_idf & est_domaine_f & ~types.isin(TYPES)

        codes_inconnus.update(
            types.loc[inconnus].dropna().tolist()
        )

        masque = est_idf & types.isin(TYPES)

        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if codes_inconnus:
    raise ValueError(
        "Codes du domaine F non prévus : "
        + ", ".join(sorted(codes_inconnus))
    )

if not blocs_retenus:
    raise ValueError("Aucune observation retenue.")

equipements = pd.concat(blocs_retenus, ignore_index=True)
del blocs_retenus

if not nettoyer_texte(equipements["AN"]).eq("2025").fillna(False).all():
    raise ValueError("L'extraction contient un millésime autre que 2025.")

print("Observations retenues :", len(equipements))

500,000 lignes examinées
1,000,000 lignes examinées
1,500,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Observations retenues : 20478


In [9]:
#Tracabilité et classification

HASH_BPE = sha256_fichier(FICHIER_BPE)
DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

equipements["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + equipements["RANG_SOURCE_J10E"].astype("string")
)

equipements["CATEGORIE_ANALYSE"] = (
    equipements["TYPEQU_ANALYSE"].map(TYPES)
)

equipements["GROUPE_CODE"] = (
    equipements["TYPEQU_ANALYSE"].str[:2]
)

equipements["GROUPE_ANALYSE"] = (
    equipements["GROUPE_CODE"].map(GROUPES)
)

equipements["SOURCE_DONNEES"] = "INSEE_BPE_2025"
equipements["STATUT_OUVERTURE_ACTUELLE"] = "NON_VERIFIE"

assert equipements["ID_OBSERVATION"].is_unique
assert equipements["GROUPE_ANALYSE"].notna().all()

display(equipements["GROUPE_ANALYSE"].value_counts())

GROUPE_ANALYSE
SPORTS               17857
CULTURE               2395
LOISIRS_PLEIN_AIR      226
Name: count, dtype: int64

In [10]:
#Harmoniser les communes

equipements["CODGEO"] = equipements["CODGEO_SOURCE"]

arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & arrondissements_paris:
    raise ValueError(
        "Le profil doit être communal, sans arrondissements parisiens."
    )

if "75056" in codes_profil:
    equipements.loc[
        equipements["CODGEO"].isin(arrondissements_paris),
        "CODGEO",
    ] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    equipements.loc[
        equipements["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

hors_profil = equipements.loc[
    ~equipements["CODGEO"].isin(codes_profil)
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "equipements_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )

    raise ValueError(
        "Résoudre les communes non appariées avant de poursuivre."
    )

Observations hors profil : 0


In [11]:
#Noms, adresses et répétitions de SIRET

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "LATITUDE", "LONGITUDE",
    "CAPACITE", "INDIC_CAPA",
    "NBEQUIDENT", "INDIC_NBEQUIDENT",
    "MULTIPLEXE",
]

for colonne in colonnes_facultatives:
    if colonne not in equipements.columns:
        equipements[colonne] = pd.Series(
            pd.NA,
            index=equipements.index,
            dtype="string",
        )

equipements["NOM_AFFICHAGE"] = (
    nettoyer_texte(equipements["NOMRS"])
    .fillna(nettoyer_texte(equipements["CNOMRS"]))
)

parties_adresse = equipements[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

equipements["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

equipements["SIRET_NETTOYE"] = nettoyer_texte(
    equipements["SIRET"]
)

siret_conforme = (
    equipements["SIRET_NETTOYE"].str.fullmatch(r"\d{14}", na=False)
    & equipements["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

equipements["SIRET_TYPE_REPETE"] = (
    siret_conforme
    & equipements.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = equipements.loc[
    equipements["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes_a_examiner.csv",
)

print("Observations avec SIRET et type répétés :", len(repetitions))
print("Aucune suppression automatique.")

Observations avec SIRET et type répétés : 57
Aucune suppression automatique.


In [12]:
#Contrôler les coordonnées

equipements["LATITUDE_NUM"] = convertir_nombre(
    equipements["LATITUDE"]
)

equipements["LONGITUDE_NUM"] = convertir_nombre(
    equipements["LONGITUDE"]
)

coordonnees_presentes = (
    equipements["LATITUDE_NUM"].notna()
    & equipements["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    equipements["LATITUDE_NUM"].between(48.0, 49.3)
    & equipements["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

equipements["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

equipements.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

equipements.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    equipements.groupby(
        ["GROUPE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,GROUPE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,CULTURE,PLAUSIBLES_A_VERIFIER,2395
1,LOISIRS_PLEIN_AIR,ABSENTES_OU_NON_NUMERIQUES,2
2,LOISIRS_PLEIN_AIR,PLAUSIBLES_A_VERIFIER,224
3,SPORTS,ABSENTES_OU_NON_NUMERIQUES,122
4,SPORTS,PLAUSIBLES_A_VERIFIER,17735


In [13]:
#Conserver la multiplicité des équipements sportifs

est_sport_loisir = equipements["GROUPE_CODE"].isin(["F1", "F2"])

nombre_identiques = convertir_nombre(
    equipements["NBEQUIDENT"]
)

nombre_valide = (
    nombre_identiques.notna()
    & nombre_identiques.ge(0)
    & nombre_identiques.mod(1).eq(0)
).fillna(False)

concerne_nombre = nettoyer_texte(
    equipements["INDIC_NBEQUIDENT"]
).eq("1").fillna(False)

equipements["NB_EQUIPEMENTS_IDENTIQUES_NUM"] = (
    nombre_identiques.where(
        est_sport_loisir & nombre_valide & concerne_nombre
    )
)

diagnostic_multiplicite = (
    equipements.loc[est_sport_loisir]
    .groupby("CATEGORIE_ANALYSE")
    .agg(
        NB_OBSERVATIONS=("ID_OBSERVATION", "size"),
        NB_OBSERVATIONS_MULTIPLICITE_CONNUE=(
            "NB_EQUIPEMENTS_IDENTIQUES_NUM", "count"
        ),
    )
    .reset_index()
)

sauvegarder_csv(
    diagnostic_multiplicite,
    DOSSIER_INTERIM / "controle_multiplicite_sport_loisirs.csv",
)

display(diagnostic_multiplicite)

,CATEGORIE_ANALYSE,NB_OBSERVATIONS,NB_OBSERVATIONS_MULTIPLICITE_CONNUE
0,ATHLETISME,866,866
1,BAIGNADE_AMENAGEE,5,5
2,BOULES_PETANQUE,1001,1001
3,BOWLING,40,40
4,CYCLISME,58,58
5,EQUITATION,435,435
6,ESCALADE_ARTIFICIELLE,374,374
7,FITNESS_REMISE_EN_FORME,1222,1222
8,GOLF,99,99
9,GYMNASES_MULTISPORTS,2239,2239


In [14]:
#Compter par sous-catégorie et groupe

COLONNES_TYPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in TYPES.items()
}

COLONNES_GROUPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in GROUPES.items()
}

comptages = pd.crosstab(
    equipements["CODGEO"],
    equipements["TYPEQU_ANALYSE"],
)

comptages = (
    comptages.reindex(columns=list(TYPES), fill_value=0)
    .rename(columns=COLONNES_TYPES)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_types = list(COLONNES_TYPES.values())

indicateurs[colonnes_types] = (
    indicateurs[colonnes_types]
    .fillna(0)
    .astype("int64")
)

for groupe, colonne_sortie in COLONNES_GROUPES.items():
    colonnes_a_sommer = [
        COLONNES_TYPES[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    indicateurs[colonne_sortie] = (
        indicateurs[colonnes_a_sommer].sum(axis=1)
    )

COLONNE_TOTAL = "NB_OBS_SPORTS_LOISIRS_CULTURE_BPE2025"

indicateurs[COLONNE_TOTAL] = (
    indicateurs[colonnes_types].sum(axis=1)
)

assert indicateurs[COLONNE_TOTAL].sum() == len(equipements)

assert (
    indicateurs[list(COLONNES_GROUPES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

print("Comptages contrôlés.")

Comptages contrôlés.


In [15]:
#Calculer les parts utiles

total = indicateurs[COLONNE_TOTAL].replace(0, np.nan)

PARTS_GROUPES = {}
PARTS_INTRAGROUPE = {}
series_parts = {}

for code, libelle in GROUPES.items():
    nom_part = f"PART_{libelle}_J10E_PCT"

    series_parts[nom_part] = (
        indicateurs[COLONNES_GROUPES[code]]
        .div(total)
        .mul(100)
    )

    PARTS_GROUPES[code] = nom_part

for code, libelle in TYPES.items():
    nom_part = f"PART_{libelle}_DANS_GROUPE_PCT"

    total_groupe = indicateurs[
        COLONNES_GROUPES[code[:2]]
    ].replace(0, np.nan)

    series_parts[nom_part] = (
        indicateurs[COLONNES_TYPES[code]]
        .div(total_groupe)
        .mul(100)
    )

    PARTS_INTRAGROUPE[code] = nom_part

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_parts, index=indicateurs.index),
    ],
    axis=1,
)

In [16]:
#Contrôler les parts

def verifier_parts(table, colonnes, colonne_total):
    avec_observations = table[colonne_total].gt(0)

    valeurs = table.loc[avec_observations, colonnes]

    assert valeurs.notna().all().all()
    assert ((valeurs >= 0) & (valeurs <= 100)).all().all()

    assert np.allclose(
        valeurs.sum(axis=1).to_numpy(dtype=float),
        100.0,
    )

    assert table.loc[
        ~avec_observations, colonnes
    ].isna().all().all()


verifier_parts(
    indicateurs,
    list(PARTS_GROUPES.values()),
    COLONNE_TOTAL,
)

for groupe in GROUPES:
    colonnes = [
        PARTS_INTRAGROUPE[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    verifier_parts(
        indicateurs,
        colonnes,
        COLONNES_GROUPES[groupe],
    )

print("Parts des groupes et parts intragroupe contrôlées.")

Parts des groupes et parts intragroupe contrôlées.


In [17]:
#Reprendre la population au J10d

COLONNE_POPULATION = None
metadata_j10d = {}

if FICHIER_METADATA_J10D.is_file():
    with open(
        FICHIER_METADATA_J10D,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_j10d = json.load(flux)

if COLONNE_POPULATION is None:
    COLONNE_POPULATION = metadata_j10d.get("colonne_population")

if COLONNE_POPULATION is None:
    candidats = [
        colonne
        for colonne in [
            "POPULATION",
            "POPULATION_MUNICIPALE",
            "P23_POP",
            "P22_POP",
            "POP",
        ]
        if colonne in profil.columns
    ]

    if len(candidats) == 1:
        COLONNE_POPULATION = candidats[0]
    else:
        print("Candidats :", candidats)
        raise ValueError(
            "Renseigne COLONNE_POPULATION comme au J10d."
        )

if COLONNE_POPULATION not in profil.columns:
    raise ValueError("Colonne de population introuvable.")

population_par_code = (
    profil.set_index("CODGEO")[COLONNE_POPULATION]
    .pipe(convertir_nombre)
)

population = indicateurs["CODGEO"].map(population_par_code)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)

Population utilisée : POPULATION_2022


In [18]:
#Calculer les 3 densités de groupes

DENSITES = {
    COLONNES_GROUPES[code]:
        f"DENSITE_OBS_{libelle}_10000_HAB"
    for code, libelle in GROUPES.items()
}

series_densites = {
    colonne_densite: (
        indicateurs[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )
    for colonne_nombre, colonne_densite in DENSITES.items()
}

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_densites, index=indicateurs.index),
    ],
    axis=1,
)

print("Densités créées :")
print(list(DENSITES.values()))

Densités créées :
['DENSITE_OBS_SPORTS_10000_HAB', 'DENSITE_OBS_LOISIRS_PLEIN_AIR_10000_HAB', 'DENSITE_OBS_CULTURE_10000_HAB']


In [19]:
#Préparer le détail des cinémas

cinemas = equipements.loc[
    equipements["TYPEQU_ANALYSE"].eq("F303")
].copy()

capacite = convertir_nombre(cinemas["CAPACITE"])

capacite_valide = (
    capacite.notna()
    & capacite.ge(0)
    & capacite.mod(1).eq(0)
).fillna(False)

concerne_capacite = nettoyer_texte(
    cinemas["INDIC_CAPA"]
).eq("1").fillna(False)

cinemas["NB_FAUTEUILS"] = capacite.where(
    capacite_valide & concerne_capacite
)

cinemas["CAPACITE_CONNUE"] = cinemas["NB_FAUTEUILS"].notna()

cinemas["STATUT_MULTIPLEXE"] = (
    nettoyer_texte(cinemas["MULTIPLEXE"])
    .map({"0": "NON", "1": "OUI"})
    .fillna("INCONNU")
)

print("Observations de cinémas :", len(cinemas))
print("Capacités connues :", int(cinemas["CAPACITE_CONNUE"].sum()))

display(
    cinemas[
        [
            "NOM_AFFICHAGE",
            "CODGEO",
            "NB_FAUTEUILS",
            "STATUT_MULTIPLEXE",
        ]
    ].head(20)
)

Observations de cinémas : 308
Capacités connues : 308


,NOM_AFFICHAGE,CODGEO,NB_FAUTEUILS,STATUT_MULTIPLEXE
15,UGC CINE CITE LES HALLES,75056,3896,OUI
45,GRAND REX,75056,4274,NON
46,PATHE,75056,1138,NON
80,MK 2 BEAUBOURG,75056,557,NON
116,LUMINOR HOTEL DE VILLE,75056,238,NON
186,ESPACE SAINT MICHEL,75056,210,NON
187,REFLET MEDICIS 2,75056,339,NON
188,QUARTIER LATIN I,75056,164,NON
189,LE GRAND ACTION,75056,363,NON
190,ECOLES CINEMA CLUB,75056,200,NON


In [22]:
#Agréger les capacités de cinémas

capacites_communes = (
    cinemas.groupby("CODGEO")
    .agg(
        NB_OBS_CINEMAS_CAPACITE_CONNUE=(
            "NB_FAUTEUILS", "count"
        ),
        FAUTEUILS_CINEMA_SOMME_CONNUE=(
            "NB_FAUTEUILS",
            lambda serie: serie.sum(min_count=1),
        ),
    )
    .reset_index()
)

indicateurs = indicateurs.merge(
    capacites_communes,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonne_nb_cinemas = COLONNES_TYPES["F303"]

indicateurs["NB_OBS_CINEMAS_CAPACITE_CONNUE"] = (
    indicateurs["NB_OBS_CINEMAS_CAPACITE_CONNUE"]
    .fillna(0)
    .astype("int64")
)

sans_cinema = indicateurs[colonne_nb_cinemas].eq(0)

capacite_complete = (
    indicateurs["NB_OBS_CINEMAS_CAPACITE_CONNUE"]
    .eq(indicateurs[colonne_nb_cinemas])
)

# Zéro seulement lorsqu'aucun cinéma n'est recensé.
indicateurs.loc[
    sans_cinema,
    "FAUTEUILS_CINEMA_SOMME_CONNUE",
] = 0

indicateurs["FAUTEUILS_CINEMA_TOTAL_BPE2025"] = (
    indicateurs["FAUTEUILS_CINEMA_SOMME_CONNUE"]
    .where(capacite_complete)
)

indicateurs["STATUT_CAPACITE_CINEMA"] = "INCOMPLETE"

indicateurs.loc[
    capacite_complete & ~sans_cinema,
    "STATUT_CAPACITE_CINEMA",
] = "COMPLETE_DANS_EXTRACTION"

indicateurs.loc[
    sans_cinema,
    "STATUT_CAPACITE_CINEMA",
] = "AUCUN_CINEMA_DANS_EXTRACTION"

display(
    indicateurs[
        [
            "CODGEO",
            colonne_nb_cinemas,
            "NB_OBS_CINEMAS_CAPACITE_CONNUE",
            "FAUTEUILS_CINEMA_SOMME_CONNUE",
            "FAUTEUILS_CINEMA_TOTAL_BPE2025",
            "STATUT_CAPACITE_CINEMA",
        ]
    ].head(10)
)

,CODGEO,NB_OBS_CINEMAS_BPE2025,NB_OBS_CINEMAS_CAPACITE_CONNUE,FAUTEUILS_CINEMA_SOMME_CONNUE,FAUTEUILS_CINEMA_TOTAL_BPE2025,STATUT_CAPACITE_CINEMA
0,75056,77,77,68301,68301,COMPLETE_DANS_EXTRACTION
1,77001,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
2,77002,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
3,77003,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
4,77004,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
5,77005,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
6,77006,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
7,77007,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
8,77008,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION
9,77009,0,0,0,0,AUCUN_CINEMA_DANS_EXTRACTION


In [23]:
#Joindre au profil J10d

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Colonnes déjà présentes dans le profil : "
        + ", ".join(sorted(collisions))
    )

if indicateurs.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans les indicateurs.")

profil_j10e = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10e) == len(profil)
assert profil_j10e["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10e[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Colonnes avant :", len(profil.columns))
print("Colonnes après :", len(profil_j10e.columns))

Colonnes avant : 783
Colonnes après : 871


In [24]:
#Afficher les résultats

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10e.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    "NB_OBS_SPORTS_BPE2025",
    "NB_OBS_LOISIRS_PLEIN_AIR_BPE2025",
    "NB_OBS_CULTURE_BPE2025",
    "NB_OBS_CINEMAS_BPE2025",
    "FAUTEUILS_CINEMA_TOTAL_BPE2025",
    "STATUT_CAPACITE_CINEMA",
]

display(
    profil_j10e[colonnes_affichage]
    .sort_values("NB_OBS_CULTURE_BPE2025", ascending=False)
    .head(20)
)

print("Cergy et Pontoise :")

display(
    profil_j10e.loc[
        profil_j10e["CODGEO"].isin(["95127", "95500"]),
        colonnes_affichage,
    ]
)

bilan_categories = (
    equipements.groupby(
        ["TYPEQU_ANALYSE", "CATEGORIE_ANALYSE", "GROUPE_ANALYSE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    bilan_categories,
    DOSSIER_INTERIM / "bilan_categories.csv",
)

,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_SPORTS_BPE2025,NB_OBS_LOISIRS_PLEIN_AIR_BPE2025,NB_OBS_CULTURE_BPE2025,NB_OBS_CINEMAS_BPE2025,FAUTEUILS_CINEMA_TOTAL_BPE2025,STATUT_CAPACITE_CINEMA
0,75056,Paris,2113705.0,1632,0,402,77,68301,COMPLETE_DANS_EXTRACTION
754,78646,Versailles,83918.0,103,1,20,2,2359,COMPLETE_DANS_EXTRACTION
1027,93066,Saint-Denis,148907.0,170,0,20,2,1911,COMPLETE_DANS_EXTRACTION
965,92012,Boulogne-Billancourt,120205.0,100,0,19,2,1646,COMPLETE_DANS_EXTRACTION
1048,94028,Créteil,92859.0,129,3,17,3,4292,COMPLETE_DANS_EXTRACTION
984,92050,Nanterre,98119.0,154,0,15,2,2151,COMPLETE_DANS_EXTRACTION
988,92063,Rueil-Malmaison,80842.0,148,1,14,2,1377,COMPLETE_DANS_EXTRACTION
276,77288,Melun,43685.0,56,0,14,2,834,COMPLETE_DANS_EXTRACTION
719,78551,Saint-Germain-en-Laye,45286.0,70,0,13,1,742,COMPLETE_DANS_EXTRACTION
998,93005,Aulnay-sous-Bois,86360.0,97,0,13,2,3379,COMPLETE_DANS_EXTRACTION


Cergy et Pontoise :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_SPORTS_BPE2025,NB_OBS_LOISIRS_PLEIN_AIR_BPE2025,NB_OBS_CULTURE_BPE2025,NB_OBS_CINEMAS_BPE2025,FAUTEUILS_CINEMA_TOTAL_BPE2025,STATUT_CAPACITE_CINEMA
1119,95127,Cergy,69578.0,104,5,12,1,2788,COMPLETE_DANS_EXTRACTION
1220,95500,Pontoise,31623.0,91,3,9,1,272,COMPLETE_DANS_EXTRACTION


In [25]:
#Créer le dictionnaire des indicateurs

lignes_dictionnaire = []

def documenter(colonne, description, unite, denominateur=""):
    lignes_dictionnaire.append({
        "COLONNE": colonne,
        "DESCRIPTION": description,
        "UNITE": unite,
        "DENOMINATEUR": denominateur,
    })


for code, libelle in TYPES.items():
    documenter(
        COLONNES_TYPES[code],
        f"Observations BPE du type {code} : {libelle}",
        "observations",
    )

    documenter(
        PARTS_INTRAGROUPE[code],
        f"Part du type {code} dans son groupe",
        "%",
        COLONNES_GROUPES[code[:2]],
    )


for code, libelle in GROUPES.items():
    colonne_nombre = COLONNES_GROUPES[code]

    documenter(
        colonne_nombre,
        f"Observations du groupe {libelle}",
        "observations",
    )

    documenter(
        PARTS_GROUPES[code],
        f"Part du groupe {libelle} dans J10e",
        "%",
        COLONNE_TOTAL,
    )

    documenter(
        DENSITES[colonne_nombre],
        f"Densité du groupe {libelle}",
        "observations pour 10 000 habitants",
        COLONNE_POPULATION,
    )


documenter(
    COLONNE_TOTAL,
    "Total des observations sports, loisirs et culture",
    "observations",
)

documenter(
    "NB_OBS_CINEMAS_CAPACITE_CONNUE",
    "Observations de cinémas avec capacité exploitable",
    "observations",
)

documenter(
    "FAUTEUILS_CINEMA_SOMME_CONNUE",
    "Somme des capacités connues, potentiellement partielle",
    "fauteuils",
)

documenter(
    "FAUTEUILS_CINEMA_TOTAL_BPE2025",
    "Capacité totale seulement si toutes les capacités sont connues",
    "fauteuils",
)

documenter(
    "STATUT_CAPACITE_CINEMA",
    "Complétude des capacités dans l'extraction",
    "statut",
)

dictionnaire = pd.DataFrame(lignes_dictionnaire)

assert dictionnaire["COLONNE"].is_unique
assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10e.csv",
)

In [26]:
#Controles finaux

colonnes_effectifs = [
    *COLONNES_TYPES.values(),
    *COLONNES_GROUPES.values(),
    COLONNE_TOTAL,
    "NB_OBS_CINEMAS_CAPACITE_CONNUE",
]

assert len(profil_j10e) == len(profil)
assert profil_j10e["CODGEO"].is_unique
assert set(profil_j10e["CODGEO"]) == codes_profil

assert equipements["ID_OBSERVATION"].is_unique
assert equipements["CODGEO"].isin(codes_profil).all()

assert indicateurs[colonnes_effectifs].notna().all().all()
assert indicateurs[colonnes_effectifs].ge(0).all().all()

assert indicateurs[COLONNE_TOTAL].sum() == len(equipements)

assert (
    indicateurs[colonnes_types].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

for colonne in DENSITES.values():
    valeurs = indicateurs[colonne].dropna().astype(float)

    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

    assert indicateurs.loc[
        ~population_valide, colonne
    ].isna().all()

assert (
    indicateurs["NB_OBS_CINEMAS_CAPACITE_CONNUE"]
    <= indicateurs["NB_OBS_CINEMAS_BPE2025"]
).all()

incomplet = indicateurs["STATUT_CAPACITE_CINEMA"].eq("INCOMPLETE")

assert indicateurs.loc[
    incomplet,
    "FAUTEUILS_CINEMA_TOTAL_BPE2025",
].isna().all()

assert indicateurs.loc[
    ~incomplet,
    "FAUTEUILS_CINEMA_TOTAL_BPE2025",
].notna().all()

for colonne in [
    "FAUTEUILS_CINEMA_SOMME_CONNUE",
    "FAUTEUILS_CINEMA_TOTAL_BPE2025",
]:
    assert indicateurs[colonne].dropna().ge(0).all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10e[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10e),
    "OBSERVATIONS_DETAIL": len(equipements),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "OBSERVATIONS_CINEMAS": len(cinemas),
    "CINEMAS_CAPACITE_CONNUE": int(cinemas["CAPACITE_CONNUE"].sum()),
    "COMMUNES_CAPACITE_CINEMA_INCOMPLETE": int(incomplet.sum()),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_DETAIL,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,OBSERVATIONS_CINEMAS,CINEMAS_CAPACITE_CONNUE,COMMUNES_CAPACITE_CINEMA_INCOMPLETE,ETAT
0,1266,1266,20478,0,57,308,308,0,CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER


In [27]:
#Enregistrer les résultats

sauvegarder_csv(equipements, FICHIER_DETAIL)
sauvegarder_csv(cinemas, FICHIER_CINEMAS)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10e, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10e_sports_loisirs_culture_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE_BPE_2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "url_dictionnaire": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_anonymisee_dictionnaire_variables.html"
    ),
    "fichier_bpe": str(FICHIER_BPE),
    "sha256_bpe": HASH_BPE,
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "colonne_population": COLONNE_POPULATION,
    "types": TYPES,
    "groupes": GROUPES,
    "nombre_observations": len(equipements),
    "nombre_communes": len(profil_j10e),
    "nombre_observations_cinemas": len(cinemas),
    "deduplication_automatique": False,
    "parts": {
        "groupes": "Dans le total J10e",
        "sous_categories": "Dans leur groupe uniquement",
    },
    "densites": "Groupes uniquement, pour 10 000 habitants",
    "limites": [
        "Comptages d'observations, pas de sites physiques uniques.",
        "NBEQUIDENT conservé dans le détail, non utilisé pour pondérer les parts.",
        "Sources pouvant avoir des dates de référence différentes.",
        "Ouverture actuelle non vérifiée.",
        "Aucune donnée de fréquentation.",
        "Fauteuils de cinéma distincts des entrées annuelles.",
        "Total de fauteuils manquant lorsque la couverture est incomplète.",
        "D505 déjà traité au J10d et exclu de cette extraction.",
        "Coordonnées contrôlées par rectangle uniquement.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10e.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Cinémas :", FICHIER_CINEMAS)
print("Indicateurs communaux :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10e enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\sports_loisirs_culture_idf_j10e.csv
Cinémas : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\cinemas_idf_j10e.csv
Indicateurs communaux : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10e\sports_loisirs_culture_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10e.csv
J10e enregistré.
